In [1]:
import sys
sys.path.append("..")

In [2]:
import json

In [3]:
s = "[\"A\", \"B\", \"C\"]\n[\"C\", \"D\", \"E\"]"

In [4]:
s.split("\n")

['["A", "B", "C"]', '["C", "D", "E"]']

In [5]:
for sline in s.split("\n"):
    print(json.loads(sline))

['A', 'B', 'C']
['C', 'D', 'E']


In [6]:
from world_rando.coord import Coord

In [7]:
c = Coord(0,0) + Coord(10, 13)
json.dumps(c)

'[10, 13]'

In [8]:
Coord(*json.loads(json.dumps(c)))

c(10,13)

In [9]:
a = set([1,2,3,4,5])
set(json.loads(json.dumps(list(a))))

{1, 2, 3, 4, 5}

In [10]:
def set_to_j(set):
    return list(set)

def set_from_j(dct):
    print(dct)
    return set(dct)

# Unfortunately, the other end of the loads does not work...
set(json.loads(json.dumps(a, default=set_to_j)))

{1, 2, 3, 4, 5}

In [11]:
json.dumps("a\nb\nc")

'"a\\nb\\nc"'

In [12]:
from typing import Dict, Iterator, MutableMapping, TypeVar, Iterable, Optional

In [13]:
KT = TypeVar("KT")
VT = TypeVar("VT")

In [14]:
class ActionException(Exception):
    def __init__(self, message):
        super().__init__(message)

In [15]:
from abc import ABC, abstractmethod

In [16]:
# The nice thing about this is that different ObjectManagers can use different interfaces for the same underlying events!
class EventInterface(ABC):

    def __init__(self) -> None:
        pass

    @abstractmethod
    def write_event(self, event) -> None:
        pass

    @abstractmethod
    def read_events(self) -> [dict[str, str]]:
        pass

    @abstractmethod
    def events_iter(selt) -> Iterator[dict[str, str]]:
        pass

# The most basic interface, simply represent the events in a list
class EventList(EventInterface):

    def __init__(self) -> None:
        self.events = []

    def write_event(self, event) -> None:
        self.events.append(event)

    def read_events(self) -> [dict[str, str]]:
        return self.events

    def events_iter(self) -> Iterator[dict[str, str]]:
        for e in self.events:
            yield e

# Writing the events to a file
#TODO: this won't work on everything
# For some types, json.loads(json.dumps(s)) != s
#TODO: this is not actually a JSON file!
# No comma delimiters between objects
# But, can't write the JSON file in an online way since the opening and closing brackets will not work properly
class EventFile(EventInterface):

    # log_file should already be open(x, "rw")
    def __init__(self, log_path) -> None:
        # Leave the file open. If the program writes many events, opening and closing the file will create significant overhead.
        self.file = open(log_path, "w+")

    def write_event(self, event) -> None:
        self.file.write(json.dumps(event) + "\n")
        #self.file.flush() # I don't think it's necessary to flush the buffer for every event

    def read_events(self) -> [dict[str, str]]:
        events = []
        self.file.seek(0)
        for line in self.file.readlines():
            event = json.loads(line)
            events.append(event)
        # Return to the end of the file
        self.file.seek(0, 2)
        return events

    def events_iter(self) -> Iterator[dict[str, str]]:
        self.file.seek(0)
        for line in self.file.readlines():
            event = json.loads(line)
            yield event
        # Return to the end of the file
        self.file.seek(0, 2)

    # This breaks the object, of course
    def close(self):
        self.file.close()

#TODO: JSONify that adds list brackets at the start and end of the filet and commas at the end of each line.

In [17]:
# This one simply stores the events in a list
# It is possible to create an object with the same interface that stores the events in a file
class ObjectManager(object):

    def __init__(self, event_interface, ei_from=None) -> None:
        self.objects = {}
        self.ei = event_interface
        # Optionally replay events from a different EI
        if ei_from:
            for event in ei_from.events_iter():
                self.replay(event)

    # 'info' is a generic field to include other information needed to properly reconstruct the object
    # Logging classes will specify what fields info must have
    def create(self, object, type : str, info) -> None:
        if object.name in self.objects:
            raise ActionException(f"Object named {object.name} already exists!")
        self.objects[object.name] = object
        event = {
            "action" : "create",
            "name" : object.name,
            "parent" : object.parent,
            "interpretation" : object.interpretation,
            "type" : type,
            "info" : info
        }
        self.ei.write_event(event)

    def event(self, event):
        self.ei.write_event(event)

    def replay(self, event):
        # If it's a create action then we need to create a new object
        if event["action"] == "create":
            # Constructor for the new object will record the necessary "create" event
            type_dict[event["type"]](self, event["name"], event["parent"], event["interpretation"], event["info"])
        # Ticks are merely for timing information, and do not affect Python World replay
        elif event["action"] == "tick":
            pass
        else:
            # Use the object dict to figure out which object should replay the event
            self.objects[event["name"]].replay(event)

    # Ticks are used to control the JS World replay
    # An algorithm can introduce ticks (for example) once per iteration
    # In JSWorld, the ticks are used for two things:
    # Step Mode can either be event-by-event or tick-by-tick (tick-by-tick is default)
    # Play Mode uses the tick speed annotation to know how long to pause during the tick.
    def tick(self, speed):
        tick = {
            "action": "tick",
            "speed": speed,
        }
        self.ei.write_event(tick)


In [18]:
# Treats it like a tuple and converts to list, so this operation is identity
# Still useful to write it like this for later, though
coord_encoder = lambda x: x
# Convert back from list using the Coord constructor
coord_decoder = lambda x: Coord(*x)
set_encoder = lambda x: list(x)
set_decoder = lambda x: set(x)
identity = lambda x: x

# If f is an encoder for X then setf_encoder(f) is an encoder for Set(X)
# Convert the set of encoded values to a list
setf_encoder = lambda encode: (lambda x: list([encode(y) for y in x]))
# Similarly, setf_decoder(f) is a decoder for Set(X) if f is a decoder for X
# Convert the list of decoded values to a set
setf_decoder = lambda decode: (lambda x: set([decode(y) for y in x]))
# In some cases, this is unnecessary...
# If the set is a LoggingSet, it only has to create individual events
# The LoggingSet knows the underlying type, and so LoggingSet events only need to use the decoder for the element values
# The set itself will be built gradually during event replay, never serialized instantaneously.
# However, if the set is not a LoggingSet, then we need this.
# If the set is not a LoggingSet, then changes to the set are not tracked by the logger (obviously)
# If you have (for example):
# d: LoggingDict[int, set[int]]
# Then
# d[7] = set([3])
# d[7] = set([4,5])
# will produce two events, whereas
# d[7].add(6)
# will not produce an event
# So only in cases where the inner dict data are immutable, the log will correctly reproduce the data structure
# If they are, you don't necessarily have to go to the trouble of making a LoggingDict[int, LoggingSet]
# You will get a less detailed log, but sometimes that is a good thing.
# To support this, we need a way to write down the types to allow for different encoders
# While the best solution would be to have an actual parser for the types, the number of types that need to be supported is not high: hardcoding them is simpler

# For this to work properly, it MUST be the case that decode(json.loads(json.dumps(encode(x)))) == x

# generic L type to allow subloggers to do the encoding
# for example, if you want a LoggingDict[int, LoggingSet[int]]
# Then 
# a = LoggingDict("A")
# a[7] = LoggingSet([3,4], "B")
# First creates an empty LoggingSet and adds two items to it (using the encoder based on the type str passed to the LoggingSet constructor
# Then creates a LoggingDict and puts the dict in there
# The "value" of the LoggingDict is a LoggingSet (and the event stores a pointer to this set)
#TODO: this is an issue because we actually want the pointer relationship to match in the replay
# The "encoder" for L types should access the ObjectManager's table
# Then the appropriate value will be fetched (for example, a[7] = ObjectManager.objects["B"])
# -> Need an "encoding" keyword for an object pointer
# e.g. LoggingSet("B", ...) encodes to "Pointer(B)"
# -> Need a "decoder" that parses "Pointer(B)" to ObjectManager.objects["B"]
# This is generic for all LoggingX types

# Technically this can be done too...
# But the encoder only has to work on individual events...
# dict_encoder = lambda encode_key, encode_value: (lambda d: {encode_key(x) : encode_value(y) for x, y in d})

#NOTE: forces dicts to be "well-typed" i.e. only have one type of keys and one type of values
#NOTE: does not actually CHECK that dicts are well-typed, and in fact dicts that are not well-typed will not be reconstructed properly

encoders = {
    "Set": set_encoder, # Generic encoder for sets of jsonables
    "Set(Coord)": setf_encoder(coord_encoder)
}
decoders = {
    "Coord": coord_decoder,
    "Set": set_decoder,
    "Set(Coord)": setf_decoder(coord_decoder)
}

# Generic encoder / decoder functions that depend on the type
def log_encode(x, type, manager):
    if type == "LPointer":
        return f"Pointer({x.name})"
    else:
        if type in encoders:
            encoder = encoders[type]
            return encoder(x)
        # If there's no encoder (for example, int), then the value does not need to change
        else:
            return x

def log_decode(x, type, manager):
    # Use the manager to follow the pointer and return the correct object
    # Nested logging pointers will take care of themselves
    if type == "LPointer":
        # Ugly way to find the name inside of the parens
        name = x.split("(")[1][:-1]
        # Trust that the object with the right name will have been created since the events are processed in order
        return manager.objects[name]
    else:
        if type in decoders:
            decoder = decoders[type]
            return decoder(x)
        else:
            return x

In [20]:
coord_set = set([Coord(1,5), Coord(3,6), Coord(2,7)])
reencode = log_decode(json.loads(json.dumps(log_encode(coord_set, "Set(Coord)", None))), "Set(Coord)", None)
assert reencode == coord_set

In [21]:
# https://stackoverflow.com/questions/61112684/how-to-subclass-a-dictionary-so-it-supports-generic-type-hints
#TODO: how to handle Coord / Tuples?
#TODO: plan for Coord (and others) is this:
# 1. Change __init__ so that it uses kwargs for ALL the non-dict stuff
# name, parent, interpretation, etc.
# Notably, this prevents the dict(a=1, b=2) syntax from working properly
# Why would you use that syntax?
# 2. Change ObjectManager so that it passes the event dict as the kwargs
# 3. Add kwargs from: str and to: str to indicate the type of mapping represented by the dict
# 4. Create an "encoders" dict that maps from type strs to encoding functions that are called on the data before json.dumps
# In this case, LogDict will call the encoder when creating the event so that the ObjectManager can json.dumps it correctly
# LogDict will have both a key encoder and a value encoder
# 5. Create a "decoders" dict that maps from type strs to encoding functions that are called on the data AFTER json.loads
# In this case, LogDict will call the decoder when replaying an event
# LogDict will have both a key decoder and a value decoder
#TODO: what about Set(Coord)?
# I guess, can specify the encoder manually...
class LogDict(MutableMapping[KT, VT]):

    def __init__(self, log : Optional[ObjectManager], name, parent, interpretation, info, dictionary=None, /, **kwargs) -> None:
        # info should contain "key_type" and "value_type" strs that index the encoders/decoders dicts
        self.name = name
        self.parent = parent
        self.interpretation = interpretation
        self.key_type = info["key_type"]
        self.value_type = info["value_type"]
        if log is not None:
            log.create(self, "dict", info)
        self.log = log
        # Save the ID in order to log correctly
        self.data: Dict[KT, VT] = {}
        if dictionary is not None:
            self.update(dictionary)
        if kwargs:
            print(kwargs)
            self.update(kwargs)

    def __contains__(self, key: VT) -> bool:
        return key in self.data

    def __delitem__(self, key: KT) -> None:
        if self.log is not None:
            event = {
                "name": self.name,
                "action": "deleteitem",
                "key": log_encode(key, self.key_type, self.log),
            }
            self.log.event(event)
        del self.data[key]

    def __getitem__(self, key: KT) -> VT:
        if key in self.data:
            return self.data[key]
        raise KeyError(key)

    def __len__(self) -> int:
        return len(self.data)

    def __iter__(self) -> Iterator[KT]:
        return iter(self.data)

    def __setitem__(self, key: KT, value: VT) -> None:
        if self.log is not None:
            event = {
                "name": self.name,
                "action": "setitem",
                "key": log_encode(key, self.key_type, self.log),
                "value": log_encode(value, self.value_type, self.log),
            }
            self.log.event(event)
        self.data[key] = value

    @classmethod
    def fromkeys(cls, iterable: Iterable[KT], value: VT) -> "LogDict":
        d = cls()
        for key in iterable:
            d[key] = value
        return d

    #TODO: some way to log a batch rather than logging each change separately?
    def update(self, other=(), /, **kwds) -> None:
        if isinstance(other, abc.Mapping):
            for key in other:
                # Use self[key] so that it will be logged through __setitem__
                #self.data[key] = other[key]
                self[key] = other[key]
        elif hasattr(other, "keys"):
            for key in other.keys():
                #self.data[key] = other[key]
                self[key] = other[key]
        else:
            for key, value in other:
                #self.data[key] = value
                self[key] = value
        for key, value in kwds.items():
            #self.data[key] = value
            self[key] = value

    def replay(self, event):
        match event["action"]:
            case "deleteitem":
                key = log_decode(event["key"], self.key_type, self.log)
                del self[event["key"]]
            case "setitem":
                key = log_decode(event["key"], self.key_type, self.log)
                value = log_decode(event["value"], self.value_type, self.log)
                # Note: this will also create an event for the event replay, since it calls __setitem__
                self[key] = value
            case _:
                raise ActionException("Action Not Found")

    def __repr__(self):
        return f"L{self.data.__repr__()}"

In [22]:
type_dict = {
    "dict" : LogDict
}

In [23]:
#type(type_dict["dict"])

In [24]:
#class JSONFormatter(logging.Formatter):
#    def format(self, log_entry):

In [25]:
#inter1 = EventList()
inter1 = EventFile("../output/test.events")
object_manager = ObjectManager(inter1)

In [26]:
l = LogDict(object_manager, "d1", None, None, {"key_type": "int", "value_type": "int"})
l2 = LogDict(object_manager, "d2", "d1", None, {"key_type": "int", "value_type": "int"})

In [27]:
l[0] = 1
l2[1] = 3
l[1] = 2
l2[4] = 7

In [28]:
#object_manager.ei.events

In [29]:
inter2 = EventList()
object_manager2 = ObjectManager(inter2, inter1)

In [30]:
object_manager2.ei.events

[{'action': 'create',
  'name': 'd1',
  'parent': None,
  'interpretation': None,
  'type': 'dict',
  'info': {'key_type': 'int', 'value_type': 'int'}},
 {'action': 'create',
  'name': 'd2',
  'parent': 'd1',
  'interpretation': None,
  'type': 'dict',
  'info': {'key_type': 'int', 'value_type': 'int'}},
 {'name': 'd1', 'action': 'setitem', 'key': 0, 'value': 1},
 {'name': 'd2', 'action': 'setitem', 'key': 1, 'value': 3},
 {'name': 'd1', 'action': 'setitem', 'key': 1, 'value': 2},
 {'name': 'd2', 'action': 'setitem', 'key': 4, 'value': 7}]

In [31]:
object_manager.objects["d1"].data == object_manager2.objects["d1"].data

True

In [245]:
l[5] = 6

In [246]:
inter1.read_events()

[{'action': 'create',
  'name': 'd1',
  'parent': None,
  'interpretation': None,
  'type': 'dict',
  'info': {'key_type': 'int', 'value_type': 'int'}},
 {'action': 'create',
  'name': 'd2',
  'parent': 'd1',
  'interpretation': None,
  'type': 'dict',
  'info': {'key_type': 'int', 'value_type': 'int'}},
 {'name': 'd1', 'action': 'setitem', 'key': 0, 'value': 1},
 {'name': 'd2', 'action': 'setitem', 'key': 1, 'value': 3},
 {'name': 'd1', 'action': 'setitem', 'key': 1, 'value': 2},
 {'name': 'd2', 'action': 'setitem', 'key': 4, 'value': 7},
 {'name': 'd1', 'action': 'setitem', 'key': 5, 'value': 6}]

In [278]:
# Testing basic encoding
#TODO: test it with event file
event_interface = EventList()
manager = ObjectManager(event_interface)
l_c = LogDict(manager, "coord_dict", None, None, {"key_type": "Coord", "value_type": "Set(Coord)"})
l_c[Coord(3,5)] = set([Coord(2,7), Coord(3,4)])
m2 = ObjectManager(EventList(), manager.ei)
assert m2.objects["coord_dict"][Coord(3,5)] == l_c[Coord(3,5)]
#m2.objects["coord_dict"].data

In [277]:
# Testing pointer encoding
manager = ObjectManager(EventList())
dict_layer_1 = LogDict(manager, "dl1", None, None, {"key_type": "Coord", "value_type": "LPointer"})
dict_layer_2 = LogDict(manager, "dl2", None, None, {"key_type": "Coord", "value_type": "Coord"})
dict_layer_1[Coord(2,3)] = dict_layer_2
dict_layer_1[Coord(2,3)][Coord(3,4)] = Coord(4,5)
m2 = ObjectManager(EventList(), manager.ei)
assert m2.objects["dl1"][Coord(2,3)] == m2.objects["dl2"]
assert m2.objects["dl1"][Coord(2,3)][Coord(3,4)] == dict_layer_1[Coord(2,3)][Coord(3,4)]
# Make sure the "pointer" part is working
m2.objects["dl2"][Coord(5,6)] = Coord(6,7)
assert m2.objects["dl1"][Coord(2,3)][Coord(5,6)] == Coord(6,7)

In [ ]:
#TODO:
# X Objects have Names rather than IDs (names must be unique or an exception is thrown)
# X Dicts have data type annotations for serialization / deserialization
    # Need a custom serialization dict for the "from json" and "to_json" stuff
# X Parents - Data structures can refer to other data structures by name
# X Interpretations - A data structure e.g. LogDict can serve as (for example) a:
#   - Distance Field (over a parent space)
#   - Partition (over a parent space)
#   - etc.
# X Data structure should include annotation on the "interpretation" so that JS World can display its information properly

In [279]:
# Some Examples
# world_rando.concrete_map.Path corresponds to an edge in abstract_map
# Type: [coord] (at least, this is all that must be preserved for JSWorld)
# Parent: cmap
# (corresponds to the edge on parent.parent (which is the amap)
# node names are preserved referents
# Interpretation: Path

# world_rando.concrete_map.merging_partition
# Type: coord -> {coord} (this is an issue for JSON)
# Parent: cmap
# Interpretation: "basic" partition (keys are arbitrary, values are parts of the partition)

In [ ]:
# Core features:
# Show each generated data structure by name
# Show the ancestry relationship of the data structures
# Display shows each event change as it happens
#
# Desired features:
# Collapsible "windows" for each thing being displayed (clickable "eye")
# If display is delegated to the parent, a clickable eye that toggles visibility
# Show newly-created or newly-altered data structures in a different color
# Playback speeds (fast, slow, skip to the end, step once, etc.)
# Mousing over highlights elements that correspond to each other
#   - For example, mousing over a path on a cmap highlights the corresponding edge on the amap
#   - Mousing over a node on the cmap highlights the node on the amap
#   - etc.